# LAB 2 – Bài 2: Phân loại món ăn VinaFood21 bằng GoogLeNet

## Phiên bản bài giảng dành cho sinh viên

Trong bài này, chúng ta sử dụng **GoogLeNet** để phân loại ảnh món ăn Việt Nam thuộc 21 lớp.

Notebook không chỉ cung cấp code chạy được mà còn giải thích:

- Vì sao GoogLeNet sử dụng Inception module.
- Vai trò của convolution 1×1.
- Vai trò của hai auxiliary classifiers.
- Cách chia train, validation và test đúng.
- Cách dùng data augmentation hợp lý.
- Cách lưu checkpoint tốt nhất và early stopping.
- Cách phân tích lỗi bằng confusion matrix và ảnh dự đoán sai.

## Mục tiêu học tập

Sau khi hoàn thành bài, sinh viên có thể:

1. Đọc dữ liệu ảnh theo cấu trúc thư mục bằng `ImageFolder`.
2. Tạo transform riêng cho train và validation/test.
3. Tách validation có phân tầng theo lớp.
4. Giải thích cấu trúc GoogLeNet và Inception module.
5. Huấn luyện GoogLeNet với auxiliary loss.
6. Đánh giá bằng accuracy, precision, recall, macro F1 và top-3 accuracy.
7. Phân tích lớp khó và các cặp lớp thường bị nhầm.
8. Viết một thí nghiệm có thể tái lập.

> **Nguyên tắc quan trọng:** Tập test chỉ được dùng sau khi đã hoàn tất việc lựa chọn mô hình.

# 1. Cấu trúc dữ liệu

Dữ liệu phải được tổ chức theo dạng:

```text
VinaFood21/
├── train/
│   ├── Banh_chung/
│   │   ├── image_001.jpg
│   │   └── ...
│   ├── Banh_mi/
│   └── ...
└── test/
    ├── Banh_chung/
    ├── Banh_mi/
    └── ...
```

Mỗi thư mục con là tên của một lớp.

Trên Kaggle, đường dẫn thường là:

```text
/kaggle/input/vinafood21/VinaFood21
```

Ở máy cá nhân, có thể đặt thư mục `VinaFood21` cùng cấp với notebook.

# 2. Cài đặt thư viện

Chỉ chạy khi môi trường chưa có các thư viện cần thiết:

```python
%pip install torch torchvision scikit-learn matplotlib pillow
```

Không cần chạy lại lệnh cài đặt ở mọi lần mở notebook.

In [ ]:
# Bỏ dấu # ở dòng dưới nếu cần cài thư viện.
# %pip install torch torchvision scikit-learn matplotlib pillow

# 3. Import thư viện và thiết lập tính tái lập

## Vì sao cần random seed?

Các yếu tố sau có tính ngẫu nhiên:

- Trọng số ban đầu.
- Thứ tự ảnh sau khi shuffle.
- Data augmentation.
- Một số phép toán trên GPU.

Đặt seed giúp kết quả dễ kiểm chứng hơn. Tuy nhiên, kết quả giữa hai loại GPU hoặc hai phiên bản thư viện vẫn có thể chênh lệch nhỏ.

In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms


SEED = 42


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Hai dòng dưới ưu tiên tính tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Thiết bị:", device)
print("PyTorch:", torch.__version__)

# 4. Cấu hình thí nghiệm

Tất cả siêu tham số được đặt tại một vị trí để:

- Dễ sửa.
- Dễ ghi vào báo cáo.
- Tránh các giá trị bị rải rác trong notebook.

## Lựa chọn ban đầu

- Ảnh đầu vào: 224×224, đúng kích thước phổ biến của GoogLeNet.
- Batch size: 32, có thể giảm xuống 16 nếu GPU thiếu bộ nhớ.
- AdamW: dễ sử dụng và có weight decay.
- Auxiliary loss weight: 0.3, theo cách dùng phổ biến của GoogLeNet.
- Validation ratio: 15% dữ liệu train.

In [ ]:
NUM_CLASSES = 21
INPUT_SIZE = 224
RESIZE_SIZE = 256

BATCH_SIZE = 32
MAX_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

VALIDATION_RATIO = 0.15
AUXILIARY_LOSS_WEIGHT = 0.3
EARLY_STOPPING_PATIENCE = 4

NUM_WORKERS = 2 if os.name != "nt" else 0
USE_MIXED_PRECISION = device.type == "cuda"

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/googlenet_output")
else:
    OUTPUT_DIR = Path("./googlenet_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "best_googlenet_vinafood21.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
TEST_RESULT_PATH = OUTPUT_DIR / "test_result.json"

print("Output directory:", OUTPUT_DIR.resolve())
print("Mixed precision:", USE_MIXED_PRECISION)

# 5. Tự động tìm thư mục dữ liệu

Không nên viết cứng một đường dẫn chỉ đúng trên máy cá nhân.

Đoạn code dưới đây thử một số vị trí thường gặp. Sinh viên có thể đặt `MANUAL_DATA_ROOT` nếu dữ liệu nằm ở vị trí khác.

In [ ]:
# Ví dụ:
# MANUAL_DATA_ROOT = Path("/content/VinaFood21")
MANUAL_DATA_ROOT = None


def find_data_root(manual_root=None):
    candidates = []

    if manual_root is not None:
        candidates.append(Path(manual_root))

    candidates.extend(
        [
            Path("/kaggle/input/vinafood21/VinaFood21"),
            Path("/kaggle/input/vinafood21"),
            Path("./VinaFood21"),
            Path("./vinafood21/VinaFood21"),
            Path("./data/VinaFood21"),
            Path("./dataset/VinaFood21"),
            Path("../VinaFood21"),
        ]
    )

    for candidate in candidates:
        train_dir = candidate / "train"
        test_dir = candidate / "test"

        if train_dir.is_dir() and test_dir.is_dir():
            return candidate

    checked_paths = "\n".join(f"- {path.resolve()}" for path in candidates)

    raise FileNotFoundError(
        "Không tìm thấy cấu trúc VinaFood21/train và VinaFood21/test.\n"
        "Các vị trí đã kiểm tra:\n"
        f"{checked_paths}\n\n"
        "Hãy sửa biến MANUAL_DATA_ROOT."
    )


DATA_ROOT = find_data_root(MANUAL_DATA_ROOT)
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("Data root:", DATA_ROOT.resolve())
print("Train directory:", TRAIN_DIR.resolve())
print("Test directory:", TEST_DIR.resolve())

# PHẦN A – DỮ LIỆU

# 6. Transform cho train và validation/test

## Vì sao transform của train khác validation?

Train transform có phép biến đổi ngẫu nhiên để tạo ra nhiều phiên bản của cùng một ảnh. Đây gọi là **data augmentation**.

Mục tiêu:

- Giảm overfitting.
- Giúp mô hình bền vững hơn với thay đổi góc nhìn, vị trí và ánh sáng.
- Không cần tạo thêm file ảnh trên ổ đĩa.

Validation và test phải dùng phép biến đổi xác định, nếu không kết quả sẽ thay đổi ngẫu nhiên giữa các lần đánh giá.

## Vì sao dùng ImageNet mean và standard deviation?

Đây là bộ thông số chuẩn thường dùng cho mô hình CNN nhận ảnh RGB 224×224. Nó cũng tương thích khi sinh viên mở rộng sang transfer learning với trọng số ImageNet.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

normalize = transforms.Normalize(
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
)

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            INPUT_SIZE,
            scale=(0.75, 1.0),
            ratio=(0.90, 1.10),
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.15,
        ),
        transforms.ToTensor(),
        normalize,
    ]
)

evaluation_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        normalize,
    ]
)

print(train_transform)

# 7. Sử dụng `ImageFolder`

`torchvision.datasets.ImageFolder` tự động:

- Tìm các thư mục lớp.
- Gán chỉ số số nguyên cho từng lớp.
- Đọc đường dẫn ảnh.
- Trả về cặp `(image, label)`.

Vì các ảnh sau transform đã là Tensor cùng kích thước, `DataLoader` mặc định có thể tự ghép batch. Không cần viết `collate_fn`.

In [ ]:
# Hai dataset cùng đọc thư mục train nhưng dùng transform khác nhau.
# Đây là điểm quan trọng: validation không được dùng augmentation ngẫu nhiên.
train_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

validation_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=evaluation_transform,
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=evaluation_transform,
)

print("Số ảnh trong thư mục train:", len(train_full_dataset))
print("Số ảnh test:", len(test_dataset))
print("Số lớp:", len(train_full_dataset.classes))
print("Tên lớp:", train_full_dataset.classes)

# 8. Kiểm tra ánh xạ lớp

Train và test phải dùng cùng một ánh xạ:

```text
tên lớp → chỉ số
```

Nếu ánh xạ khác nhau, mô hình có thể dự đoán đúng chỉ số nhưng báo sai tên món ăn.

In [ ]:
if train_full_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError(
        "Ánh xạ class_to_idx của train và test không giống nhau. "
        "Hãy kiểm tra tên các thư mục lớp."
    )

if len(train_full_dataset.classes) != NUM_CLASSES:
    raise ValueError(
        f"NUM_CLASSES={NUM_CLASSES}, nhưng dữ liệu có "
        f"{len(train_full_dataset.classes)} lớp."
    )

class_names = train_full_dataset.classes
class_to_idx = train_full_dataset.class_to_idx

print("Ánh xạ lớp hợp lệ.")
print(class_to_idx)

# 9. Chia train và validation có phân tầng

## Vì sao dùng stratified split?

Phân tầng giúp tỷ lệ của từng lớp trong train và validation gần giống nhau.

Nếu chỉ chia ngẫu nhiên đơn giản, một lớp ít ảnh có thể xuất hiện quá ít trong validation.

## Vì sao tạo hai `ImageFolder` cho cùng thư mục train?

`Subset` chỉ giữ chỉ số, không tạo bản sao ảnh. Tuy nhiên transform nằm trong dataset gốc.

Vì vậy:

- `train_full_dataset` dùng augmentation.
- `validation_full_dataset` dùng transform xác định.
- Hai subset dùng cùng danh sách chỉ số đã chia.

In [ ]:
all_indices = np.arange(len(train_full_dataset))
all_targets = np.asarray(train_full_dataset.targets)

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=all_targets,
)

train_dataset = Subset(
    train_full_dataset,
    train_indices,
)

validation_dataset = Subset(
    validation_full_dataset,
    validation_indices,
)

print("Số ảnh train:", len(train_dataset))
print("Số ảnh validation:", len(validation_dataset))
print("Số ảnh test:", len(test_dataset))

# 10. Kiểm tra phân bố lớp

Trước khi huấn luyện, cần biết dữ liệu có mất cân bằng hay không.

Nếu một số lớp có quá ít ảnh, accuracy tổng thể có thể không phản ánh đúng chất lượng của từng lớp. Vì vậy bài này còn sử dụng macro F1.

In [ ]:
train_targets = all_targets[train_indices]
validation_targets = all_targets[validation_indices]

train_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES,
)

validation_counts = np.bincount(
    validation_targets,
    minlength=NUM_CLASSES,
)

for class_index, class_name in enumerate(class_names):
    print(
        f"{class_index:02d} {class_name:25s} | "
        f"train: {train_counts[class_index]:4d} | "
        f"validation: {validation_counts[class_index]:4d}"
    )

In [ ]:
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, train_counts)
plt.xticks(positions, class_names, rotation=90)
plt.xlabel("Lớp")
plt.ylabel("Số ảnh")
plt.title("Phân bố lớp trong train set")
plt.tight_layout()
plt.show()

# 11. Tạo DataLoader

- Train dùng `shuffle=True`.
- Validation và test dùng `shuffle=False`.
- `pin_memory=True` chỉ có lợi khi dùng GPU.
- `worker_init_fn` và `generator` hỗ trợ tính tái lập khi dùng nhiều worker.

In [ ]:
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

pin_memory = device.type == "cuda"

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

first_images, first_labels = next(iter(train_loader))

print("Batch ảnh:", first_images.shape)
print("Batch nhãn:", first_labels.shape)
print("Kiểu dữ liệu ảnh:", first_images.dtype)

# 12. Hiển thị ảnh sau khi bỏ chuẩn hóa

Sau Normalize, giá trị ảnh không còn nằm trong `[0, 1]`. Vì vậy, muốn hiển thị đúng màu cần thực hiện phép biến đổi ngược.

In [ ]:
def denormalize_image(image_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)


number_of_samples = min(8, len(first_images))

figure, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for index in range(number_of_samples):
    image = denormalize_image(first_images[index])
    image = image.permute(1, 2, 0)

    axes[index].imshow(image)
    axes[index].set_title(class_names[first_labels[index].item()])
    axes[index].axis("off")

for index in range(number_of_samples, len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

# PHẦN B – LÝ THUYẾT GOOGLENET

# 13. Vấn đề mà GoogLeNet muốn giải quyết

Một mạng sâu hơn thường học được đặc trưng tốt hơn, nhưng kéo theo:

- Nhiều tham số.
- Tốn bộ nhớ.
- Tốn thời gian tính toán.
- Gradient khó truyền về các lớp đầu.

GoogLeNet giải quyết bằng:

1. **Inception module**: xử lý nhiều kích thước đặc trưng song song.
2. **Convolution 1×1**: giảm số kênh trước các convolution lớn.
3. **Global average pooling**: giảm số lượng tham số ở cuối mạng.
4. **Auxiliary classifiers**: hỗ trợ gradient trong quá trình huấn luyện.

# 14. Inception module

Một Inception module có nhiều nhánh song song:

```text
                     ┌─ 1×1 convolution ───────────┐
Input ───────────────┼─ 1×1 → 3×3 convolution ────┤
                     ├─ 1×1 → 5×5 convolution ────┤ → Concatenate
                     └─ max pooling → 1×1 conv ───┘
```

Các nhánh học đặc trưng ở nhiều phạm vi:

- 1×1: kết hợp thông tin giữa các kênh.
- 3×3: học đặc trưng cục bộ.
- 5×5: nhìn vùng rộng hơn.
- Pooling: giữ đặc trưng nổi bật và tăng tính bất biến.

Đầu ra các nhánh được nối theo chiều channel.

# 15. Vì sao convolution 1×1 quan trọng?

Giả sử đầu vào có 192 kênh và muốn dùng 3×3 convolution tạo 128 kênh.

Nếu làm trực tiếp, số trọng số xấp xỉ:

```text
192 × 128 × 3 × 3
```

Nếu giảm từ 192 xuống 32 kênh bằng convolution 1×1 trước:

```text
192 × 32 × 1 × 1
+
32 × 128 × 3 × 3
```

Số phép tính giảm đáng kể.

Convolution 1×1 không chỉ giảm chiều; nó còn học cách kết hợp các kênh đầu vào.

# 16. Auxiliary classifiers

GoogLeNet có hai đầu ra phụ ở giữa mạng.

Trong lúc train:

```text
total loss
=
main loss
+ 0.3 × auxiliary loss 1
+ 0.3 × auxiliary loss 2
```

Mục đích:

- Đưa tín hiệu gradient đến các lớp giữa.
- Hỗ trợ mạng sâu học ổn định hơn.
- Có tác dụng regularization nhẹ.

Khi validation hoặc inference, chỉ sử dụng main logits.

# PHẦN C – XÂY DỰNG MÔ HÌNH

# 17. Tạo GoogLeNet từ `torchvision`

Bài cơ sở huấn luyện từ đầu nên dùng:

```python
weights=None
```

`torchvision` đã cài đúng:

- Conv đầu vào 3 kênh.
- Kernel 7×7, stride 2, padding 3.
- Các MaxPool có thiết lập phù hợp.
- Hai auxiliary classifiers.

Vì vậy không cần thay lại `conv1` hoặc duyệt toàn bộ MaxPool như bài cũ.

In [ ]:
def initialize_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(
            module.weight,
            mode="fan_out",
            nonlinearity="relu",
        )

        if module.bias is not None:
            nn.init.zeros_(module.bias)

    elif isinstance(module, nn.BatchNorm2d):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.01)

        if module.bias is not None:
            nn.init.zeros_(module.bias)


def create_googlenet(num_classes):
    model = models.googlenet(
        weights=None,
        num_classes=num_classes,
        aux_logits=True,
        init_weights=False,
    )

    # Khởi tạo rõ ràng để tránh phụ thuộc vào hành vi mặc định
    # giữa các phiên bản torchvision.
    model.apply(initialize_weights)

    return model


model = create_googlenet(NUM_CLASSES).to(device)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Số tham số có thể học: {trainable_parameters:,}")

# 18. Kiểm tra đầu ra train và eval

Đây là đặc điểm dễ gây nhầm của GoogLeNet trong `torchvision`.

## Khi `model.train()`

Đầu ra chứa:

- `logits`: đầu ra chính.
- `aux_logits1`: đầu ra phụ thứ nhất.
- `aux_logits2`: đầu ra phụ thứ hai.

## Khi `model.eval()`

Đầu ra là Tensor logits chính.

Luôn kiểm tra trước khi viết training loop.

In [ ]:
dummy_images = torch.randn(2, 3, INPUT_SIZE, INPUT_SIZE, device=device)

model.train()

with torch.no_grad():
    training_outputs = model(dummy_images)

print("Kiểu đầu ra khi train:", type(training_outputs))
print("Main logits:", training_outputs.logits.shape)
print("Aux logits 1:", training_outputs.aux_logits1.shape)
print("Aux logits 2:", training_outputs.aux_logits2.shape)

model.eval()

with torch.no_grad():
    evaluation_outputs = model(dummy_images)

print("\nKiểu đầu ra khi eval:", type(evaluation_outputs))
print("Evaluation logits:", evaluation_outputs.shape)

assert training_outputs.logits.shape == (2, NUM_CLASSES)
assert evaluation_outputs.shape == (2, NUM_CLASSES)

del dummy_images, training_outputs, evaluation_outputs

if device.type == "cuda":
    torch.cuda.empty_cache()

# 19. Theo dõi kích thước tensor qua GoogLeNet

Hàm dưới đây đi qua các khối chính của mạng để sinh viên thấy cách kích thước không gian giảm và số channel tăng.

In [ ]:
def trace_main_feature_shapes(model, input_size=224):
    model.eval()

    x = torch.zeros(1, 3, input_size, input_size, device=device)

    stages = [
        ("conv1", model.conv1),
        ("maxpool1", model.maxpool1),
        ("conv2", model.conv2),
        ("conv3", model.conv3),
        ("maxpool2", model.maxpool2),
        ("inception3a", model.inception3a),
        ("inception3b", model.inception3b),
        ("maxpool3", model.maxpool3),
        ("inception4a", model.inception4a),
        ("inception4b", model.inception4b),
        ("inception4c", model.inception4c),
        ("inception4d", model.inception4d),
        ("inception4e", model.inception4e),
        ("maxpool4", model.maxpool4),
        ("inception5a", model.inception5a),
        ("inception5b", model.inception5b),
        ("avgpool", model.avgpool),
    ]

    print("input".ljust(14), tuple(x.shape))

    with torch.no_grad():
        for name, layer in stages:
            x = layer(x)
            print(name.ljust(14), tuple(x.shape))

        x = torch.flatten(x, 1)
        logits = model.fc(model.dropout(x))

    print("flatten".ljust(14), tuple(x.shape))
    print("logits".ljust(14), tuple(logits.shape))


trace_main_feature_shapes(model, INPUT_SIZE)

# PHẦN D – LOSS, OPTIMIZER VÀ METRICS

# 20. CrossEntropyLoss

Mô hình trả về logits, không trả về xác suất.

`CrossEntropyLoss` đã thực hiện phép tính cần thiết bên trong. Không thêm Softmax trước loss.

Softmax chỉ dùng khi cần diễn giải xác suất ở bước dự đoán.

# 21. AdamW và weight decay

AdamW được chọn vì:

- Dễ dùng với người mới.
- Điều chỉnh learning rate riêng cho từng tham số.
- Tách weight decay khỏi cập nhật gradient của Adam.

Weight decay giúp hạn chế trọng số trở nên quá lớn và có thể giảm overfitting.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_MIXED_PRECISION,
)

print(loss_fn)
print(optimizer)

# 22. Vì sao sử dụng mixed precision?

Khi có GPU, mixed precision cho phép một số phép tính dùng số 16-bit:

- Giảm bộ nhớ.
- Có thể tăng tốc huấn luyện.
- Hữu ích với mô hình lớn và ảnh 224×224.

`GradScaler` giúp giảm nguy cơ gradient quá nhỏ.

Trên CPU, notebook tự động tắt mixed precision.

# 23. Hàm tách đầu ra của GoogLeNet

Hàm này giúp training loop xử lý rõ ràng hai trường hợp:

- Train: có main logits và auxiliary logits.
- Eval: chỉ có main logits.

In [ ]:
def unpack_googlenet_outputs(outputs):
    if isinstance(outputs, torch.Tensor):
        return outputs, None, None

    main_logits = outputs.logits
    auxiliary_logits_1 = outputs.aux_logits1
    auxiliary_logits_2 = outputs.aux_logits2

    return (
        main_logits,
        auxiliary_logits_1,
        auxiliary_logits_2,
    )

# 24. Hàm đánh giá

Hàm đánh giá trả về:

- Loss.
- Accuracy.
- Macro precision.
- Macro recall.
- Macro F1.
- Top-3 accuracy.

## Top-3 accuracy là gì?

Dự đoán được xem là đúng nếu nhãn thật nằm trong ba lớp có logits cao nhất.

Với bài toán 21 món ăn có hình thức gần nhau, top-3 accuracy cung cấp thêm góc nhìn hữu ích.

In [ ]:
def evaluate_model(
    model,
    dataloader,
    loss_fn,
    return_predictions=False,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    total_top3_correct = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                logits = model(images)
                loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3_predictions = logits.topk(
                k=min(3, NUM_CLASSES),
                dim=1,
            ).indices

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            total_top3_correct += (
                top3_predictions
                .eq(labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "top3_accuracy": total_top3_correct / total_samples,
    }

    if return_predictions:
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred

    return metrics

# PHẦN E – HUẤN LUYỆN

# 25. Hàm huấn luyện một epoch

Thứ tự chuẩn trong mỗi batch:

1. Chuyển ảnh và nhãn sang thiết bị.
2. Xóa gradient cũ.
3. Forward.
4. Tính main loss và auxiliary losses.
5. Backward.
6. Cập nhật trọng số.

Accuracy của train được tính từ main logits.

In [ ]:
def train_one_epoch(
    model,
    dataloader,
    loss_fn,
    optimizer,
    scaler,
):
    model.train()

    total_loss = 0.0
    total_main_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            outputs = model(images)

            main_logits, aux_logits_1, aux_logits_2 = (
                unpack_googlenet_outputs(outputs)
            )

            main_loss = loss_fn(main_logits, labels)
            total_batch_loss = main_loss

            if aux_logits_1 is not None:
                auxiliary_loss_1 = loss_fn(
                    aux_logits_1,
                    labels,
                )

                total_batch_loss = (
                    total_batch_loss
                    + AUXILIARY_LOSS_WEIGHT * auxiliary_loss_1
                )

            if aux_logits_2 is not None:
                auxiliary_loss_2 = loss_fn(
                    aux_logits_2,
                    labels,
                )

                total_batch_loss = (
                    total_batch_loss
                    + AUXILIARY_LOSS_WEIGHT * auxiliary_loss_2
                )

        scaler.scale(total_batch_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = main_logits.argmax(dim=1)
        batch_size = images.size(0)

        total_loss += total_batch_loss.item() * batch_size
        total_main_loss += main_loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

    return {
        "total_loss": total_loss / total_samples,
        "main_loss": total_main_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }

# 26. Checkpoint, scheduler và early stopping

Sau mỗi epoch:

1. Train trên train set.
2. Đánh giá validation set.
3. Scheduler điều chỉnh learning rate theo validation loss.
4. Lưu checkpoint nếu validation macro F1 tốt hơn.
5. Dừng sớm nếu không cải thiện trong nhiều epoch.

Test set không xuất hiện trong vòng lặp huấn luyện.

In [ ]:
history = {
    "train_total_loss": [],
    "train_main_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "validation_f1_macro": [],
    "validation_top3_accuracy": [],
    "learning_rate": [],
}

best_validation_f1 = float("-inf")
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        scaler=scaler,
    )

    validation_metrics = evaluate_model(
        model=model,
        dataloader=validation_loader,
        loss_fn=loss_fn,
    )

    scheduler.step(validation_metrics["loss"])

    current_learning_rate = optimizer.param_groups[0]["lr"]

    history["train_total_loss"].append(
        float(train_metrics["total_loss"])
    )
    history["train_main_loss"].append(
        float(train_metrics["main_loss"])
    )
    history["train_accuracy"].append(
        float(train_metrics["accuracy"])
    )
    history["validation_loss"].append(
        float(validation_metrics["loss"])
    )
    history["validation_accuracy"].append(
        float(validation_metrics["accuracy"])
    )
    history["validation_f1_macro"].append(
        float(validation_metrics["f1_macro"])
    )
    history["validation_top3_accuracy"].append(
        float(validation_metrics["top3_accuracy"])
    )
    history["learning_rate"].append(
        float(current_learning_rate)
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train total loss: {train_metrics['total_loss']:.4f} | "
        f"Train acc: {train_metrics['accuracy']:.4f} | "
        f"Val loss: {validation_metrics['loss']:.4f} | "
        f"Val acc: {validation_metrics['accuracy']:.4f} | "
        f"Val macro F1: {validation_metrics['f1_macro']:.4f} | "
        f"Val top-3: {validation_metrics['top3_accuracy']:.4f} | "
        f"LR: {current_learning_rate:.2e}"
    )

    if validation_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = validation_metrics["f1_macro"]
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "validation_metrics": {
                key: float(value)
                for key, value in validation_metrics.items()
            },
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": {
                "seed": SEED,
                "num_classes": NUM_CLASSES,
                "input_size": INPUT_SIZE,
                "batch_size": BATCH_SIZE,
                "max_epochs": MAX_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "validation_ratio": VALIDATION_RATIO,
                "auxiliary_loss_weight": AUXILIARY_LOSS_WEIGHT,
            },
        }

        torch.save(checkpoint, CHECKPOINT_PATH)

        print(
            "  -> Đã lưu checkpoint mới. "
            f"Validation macro F1 = {best_validation_f1:.4f}"
        )
    else:
        epochs_without_improvement += 1

        print(
            "  -> Chưa cải thiện. "
            f"Early stopping: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "\nDừng sớm vì validation macro F1 "
            "không còn cải thiện."
        )
        break

with HISTORY_PATH.open("w", encoding="utf-8") as file:
    json.dump(history, file, indent=2, ensure_ascii=False)

print("\nĐã lưu history tại:", HISTORY_PATH.resolve())
print("Checkpoint tốt nhất:", CHECKPOINT_PATH.resolve())

# 27. Biểu đồ train và validation loss

Nếu train loss tiếp tục giảm nhưng validation loss tăng, mô hình có thể đang overfit.

`Train total loss` bao gồm main loss và auxiliary losses. Khi so sánh trực tiếp với validation loss, nên ưu tiên xem `train main loss`.

In [ ]:
epochs_ran = range(1, len(history["train_main_loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["train_main_loss"],
    marker="o",
    label="Train main loss",
)
plt.plot(
    epochs_ran,
    history["validation_loss"],
    marker="o",
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train main loss và validation loss")
plt.grid(True)
plt.legend()
plt.show()

# 28. Biểu đồ accuracy và macro F1

Accuracy cho biết tỷ lệ đúng tổng thể.

Macro F1 xem mỗi lớp có vai trò ngang nhau. Với dữ liệu mất cân bằng, macro F1 thường phản ánh công bằng hơn.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["validation_accuracy"],
    marker="o",
    label="Validation accuracy",
)
plt.plot(
    epochs_ran,
    history["validation_f1_macro"],
    marker="o",
    label="Validation macro F1",
)
plt.plot(
    epochs_ran,
    history["validation_top3_accuracy"],
    marker="o",
    label="Validation top-3 accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Chất lượng trên validation set")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

# 29. Biểu đồ learning rate

Scheduler giảm learning rate khi validation loss không cải thiện.

Learning rate nhỏ hơn giúp mô hình tinh chỉnh trọng số cẩn thận hơn ở giai đoạn sau.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["learning_rate"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning rate theo epoch")
plt.grid(True)
plt.show()

# PHẦN F – ĐÁNH GIÁ CUỐI CÙNG

# 30. Tải lại checkpoint tốt nhất

Epoch cuối chưa chắc là epoch tốt nhất.

Mô hình được tải lại từ checkpoint có validation macro F1 cao nhất trước khi test.

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Validation macro F1 tốt nhất:",
    f"{checkpoint['validation_metrics']['f1_macro']:.4f}",
)

# 31. Chỉ bây giờ mới đánh giá test set

Test score là kết quả cuối cùng.

Không được nhìn test score rồi:

- Chọn lại epoch.
- Đổi learning rate.
- Đổi augmentation.
- Chọn kiến trúc khác.

Các quyết định đó phải dựa trên validation set.

In [ ]:
test_metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    loss_fn=loss_fn,
    return_predictions=True,
)

print("KẾT QUẢ CUỐI CÙNG TRÊN TEST SET")
print("-" * 50)
print(f"Loss:            {test_metrics['loss']:.4f}")
print(f"Accuracy:        {test_metrics['accuracy']:.4f}")
print(f"Macro precision: {test_metrics['precision_macro']:.4f}")
print(f"Macro recall:    {test_metrics['recall_macro']:.4f}")
print(f"Macro F1:        {test_metrics['f1_macro']:.4f}")
print(f"Top-3 accuracy:  {test_metrics['top3_accuracy']:.4f}")

In [ ]:
serializable_test_result = {
    key: float(value)
    for key, value in test_metrics.items()
    if key not in {"y_true", "y_pred"}
}

with TEST_RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        serializable_test_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Đã lưu test result tại:", TEST_RESULT_PATH.resolve())

# 32. Classification report

Báo cáo theo từng lớp giúp phát hiện:

- Lớp có precision thấp.
- Lớp có recall thấp.
- Lớp có F1 thấp nhất.
- Chênh lệch chất lượng giữa các món ăn.

In [ ]:
print(
    classification_report(
        test_metrics["y_true"],
        test_metrics["y_pred"],
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

# 33. Confusion matrix chuẩn hóa

- Hàng: nhãn thật.
- Cột: nhãn dự đoán.
- Mỗi hàng được chuẩn hóa về tỷ lệ.

Đường chéo càng gần 1 càng tốt. Các ô lớn ngoài đường chéo cho biết những cặp món ăn thường bị nhầm.

In [ ]:
normalized_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    normalize="true",
)

display = ConfusionMatrixDisplay(
    confusion_matrix=normalized_matrix,
    display_labels=class_names,
)

figure, axis = plt.subplots(figsize=(16, 16))
display.plot(
    ax=axis,
    values_format=".2f",
    xticks_rotation=90,
)
axis.set_title("Normalized confusion matrix")
plt.tight_layout()
plt.show()

# 34. Độ chính xác theo từng lớp

Đường chéo của confusion matrix chuẩn hóa tương đương recall của từng lớp.

Biểu đồ này giúp tìm lớp khó nhất nhanh hơn.

In [ ]:
per_class_accuracy = np.diag(normalized_matrix)
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, per_class_accuracy)
plt.xticks(positions, class_names, rotation=90)
plt.ylim(0, 1)
plt.xlabel("Lớp")
plt.ylabel("Accuracy theo lớp")
plt.title("Độ chính xác của từng lớp")
plt.tight_layout()
plt.show()

hardest_class_index = int(np.argmin(per_class_accuracy))

print("Lớp khó nhất:", class_names[hardest_class_index])
print(
    "Accuracy của lớp khó nhất:",
    f"{per_class_accuracy[hardest_class_index]:.4f}",
)

# 35. Các cặp lớp bị nhầm nhiều nhất

Đoạn code loại bỏ đường chéo rồi tìm các ô có số lượng nhầm lớn nhất.

In [ ]:
raw_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
)

confusion_without_diagonal = raw_matrix.copy()
np.fill_diagonal(confusion_without_diagonal, 0)

flat_indices = np.argsort(
    confusion_without_diagonal.ravel()
)[::-1]

print("Các cặp lớp bị nhầm nhiều nhất:")

number_printed = 0

for flat_index in flat_indices:
    true_index, predicted_index = np.unravel_index(
        flat_index,
        confusion_without_diagonal.shape,
    )

    count = confusion_without_diagonal[
        true_index,
        predicted_index,
    ]

    if count == 0:
        break

    print(
        f"- Thật: {class_names[true_index]:25s} "
        f"→ Đoán: {class_names[predicted_index]:25s} "
        f"| {count} ảnh"
    )

    number_printed += 1

    if number_printed >= 10:
        break

# 36. Hiển thị các ảnh dự đoán sai

Phân tích trực tiếp ảnh sai giúp trả lời:

- Hai món có hình thức gần giống nhau không?
- Ảnh có quá tối hoặc bị che không?
- Món ăn chỉ chiếm một phần nhỏ của ảnh không?
- Nhãn dữ liệu có khả năng sai không?

In [ ]:
def collect_misclassified_samples(
    model,
    dataloader,
    max_samples=12,
):
    model.eval()
    samples = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                logits = model(images)

            probabilities = torch.softmax(logits, dim=1)
            predictions = logits.argmax(dim=1)

            wrong_indices = (
                predictions != labels
            ).nonzero(as_tuple=True)[0]

            for index in wrong_indices:
                predicted_index = predictions[index].item()

                samples.append(
                    {
                        "image": images[index].cpu(),
                        "true_index": labels[index].item(),
                        "predicted_index": predicted_index,
                        "confidence": probabilities[
                            index,
                            predicted_index,
                        ].item(),
                    }
                )

                if len(samples) >= max_samples:
                    return samples

    return samples


misclassified_samples = collect_misclassified_samples(
    model=model,
    dataloader=test_loader,
    max_samples=12,
)

print(
    "Số ảnh sai được lấy để hiển thị:",
    len(misclassified_samples),
)

In [ ]:
if misclassified_samples:
    figure, axes = plt.subplots(3, 4, figsize=(15, 11))
    axes = axes.flatten()

    for axis, sample in zip(
        axes,
        misclassified_samples,
    ):
        image = denormalize_image(sample["image"])
        image = image.permute(1, 2, 0)

        true_name = class_names[sample["true_index"]]
        predicted_name = class_names[
            sample["predicted_index"]
        ]

        axis.imshow(image)
        axis.set_title(
            f"Thật: {true_name}\n"
            f"Đoán: {predicted_name}\n"
            f"Tự tin: {sample['confidence']:.2%}",
            fontsize=9,
        )
        axis.axis("off")

    for axis in axes[len(misclassified_samples):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy ảnh dự đoán sai.")

# 37. Dự đoán một ảnh riêng lẻ

Softmax được sử dụng ở đây để chuyển logits thành xác suất dễ diễn giải.

Hàm trả về ba lớp có xác suất cao nhất.

In [ ]:
def predict_image(
    model,
    image_path,
    transform,
    class_names,
    top_k=3,
):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            logits = model(image_tensor)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        top_probabilities, top_indices = probabilities.topk(
            k=min(top_k, len(class_names)),
            dim=1,
        )

    results = []

    for probability, class_index in zip(
        top_probabilities[0].cpu(),
        top_indices[0].cpu(),
    ):
        results.append(
            {
                "class_name": class_names[class_index.item()],
                "probability": probability.item(),
            }
        )

    return image, results


# Ví dụ:
# image_path = TEST_DIR / class_names[0] / "ten_anh.jpg"
# image, results = predict_image(
#     model,
#     image_path,
#     evaluation_transform,
#     class_names,
# )
#
# plt.figure(figsize=(6, 6))
# plt.imshow(image)
# plt.axis("off")
# plt.show()
#
# for result in results:
#     print(
#         result["class_name"],
#         f"{result['probability']:.2%}",
#     )

# PHẦN G – MỞ RỘNG: TRANSFER LEARNING

# 38. Khi nào nên dùng pretrained weights?

Huấn luyện GoogLeNet từ đầu cần:

- Nhiều dữ liệu.
- Nhiều epoch.
- GPU đủ mạnh.

Nếu dữ liệu không lớn, transfer learning thường cho kết quả tốt hơn và nhanh hơn.

Đây phải được xem là một **thí nghiệm khác**. Việc so sánh vẫn phải dựa trên validation set.

In [ ]:
# MÃ THAM KHẢO – KHÔNG CHẠY CÙNG THÍ NGHIỆM TỪ ĐẦU
#
# from torchvision.models import GoogLeNet_Weights
#
# pretrained_model = models.googlenet(
#     weights=GoogLeNet_Weights.DEFAULT,
#     aux_logits=False,
# )
#
# pretrained_model.fc = nn.Linear(
#     pretrained_model.fc.in_features,
#     NUM_CLASSES,
# )
#
# pretrained_model = pretrained_model.to(device)
#
# Có thể đóng băng backbone trong một số epoch đầu:
#
# for parameter in pretrained_model.parameters():
#     parameter.requires_grad = False
#
# for parameter in pretrained_model.fc.parameters():
#     parameter.requires_grad = True

# PHẦN H – TỔNG KẾT

# 39. Quy trình chuẩn

```text
1. Kiểm tra cấu trúc thư mục
2. Kiểm tra class_to_idx
3. Tạo train transform và evaluation transform
4. Chia train–validation có phân tầng
5. Khảo sát phân bố lớp
6. Tạo DataLoader
7. Xây dựng GoogLeNet
8. Kiểm tra kiểu đầu ra train/eval
9. Huấn luyện với auxiliary losses
10. Đánh giá validation sau mỗi epoch
11. Lưu checkpoint theo validation macro F1
12. Early stopping
13. Tải lại checkpoint tốt nhất
14. Test đúng một lần
15. Phân tích confusion matrix và ảnh sai
```

# 40. Các lỗi thường gặp

## Lỗi 1: Dùng test set sau mỗi epoch

Test set bị rò rỉ vào quá trình lựa chọn mô hình.

## Lỗi 2: Dùng augmentation cho validation

Kết quả validation thay đổi ngẫu nhiên và khó so sánh.

## Lỗi 3: Train và test có `class_to_idx` khác nhau

Chỉ số dự đoán không còn tương ứng cùng tên lớp.

## Lỗi 4: Bỏ qua auxiliary losses khi train

Không tận dụng đầy đủ thiết kế của GoogLeNet.

## Lỗi 5: Cố đọc `.logits` khi `model.eval()`

Ở chế độ eval, `torchvision` thường trả về Tensor trực tiếp.

## Lỗi 6: Thêm Softmax trước CrossEntropyLoss

Không cần thiết và kém ổn định số học hơn.

## Lỗi 7: Lưu checkpoint nhưng không tải lại

Epoch cuối chưa chắc là epoch tốt nhất.

## Lỗi 8: Chỉ báo accuracy

Có thể che giấu các lớp có chất lượng rất thấp.

## Lỗi 9: Nộp notebook chưa chạy

Giảng viên không thể kiểm tra output, thời gian huấn luyện và kết quả thật.

# 41. Câu hỏi tự kiểm tra

1. Inception module có những nhánh chính nào?
2. Convolution 1×1 giúp giảm chi phí tính toán như thế nào?
3. Auxiliary classifiers được sử dụng ở giai đoạn nào?
4. Tại sao validation không dùng random crop?
5. Tại sao cần kiểm tra `class_to_idx`?
6. Macro F1 khác accuracy ở điểm nào?
7. Khi nào top-3 accuracy hữu ích?
8. Tại sao checkpoint tốt nhất phải dựa trên validation?
9. `model.train()` và `model.eval()` làm thay đổi đầu ra GoogLeNet ra sao?
10. Dấu hiệu nào cho thấy mô hình đang overfit?

# 42. Bài tập mở rộng

## Bài 1 – So sánh augmentation

So sánh:

- Chỉ resize và crop.
- Thêm horizontal flip.
- Thêm color jitter.

Chọn cấu hình bằng validation macro F1.

## Bài 2 – So sánh auxiliary loss weight

Thử:

```text
0.0, 0.2, 0.3, 0.5
```

Phân tích tốc độ hội tụ.

## Bài 3 – So sánh optimizer

So sánh AdamW và SGD momentum.

## Bài 4 – Transfer learning

Dùng trọng số ImageNet và so sánh với huấn luyện từ đầu.

## Bài 5 – Xử lý mất cân bằng

Thử class weights hoặc `WeightedRandomSampler`.

## Bài 6 – Phân tích lỗi

Chọn ba cặp lớp bị nhầm nhiều nhất và giải thích nguyên nhân bằng ảnh minh họa.

## Bài 7 – Báo cáo thí nghiệm

Báo cáo phải có:

1. Cấu trúc dữ liệu.
2. Cấu hình thí nghiệm.
3. Kiến trúc và vai trò Inception.
4. Biểu đồ huấn luyện.
5. Validation tốt nhất.
6. Test cuối cùng.
7. Confusion matrix.
8. Ảnh dự đoán sai.
9. Nhận xét và hướng cải tiến.

# Checklist trước khi nộp

- [ ] Đã chạy `Restart & Run All`.
- [ ] Không có ô code báo lỗi.
- [ ] Đã kiểm tra tên 21 lớp.
- [ ] Train và test có cùng `class_to_idx`.
- [ ] Validation không dùng augmentation ngẫu nhiên.
- [ ] Không dùng test để chọn mô hình.
- [ ] Có output từng epoch.
- [ ] Có biểu đồ loss và metrics.
- [ ] Có checkpoint tốt nhất.
- [ ] Có test result cuối cùng.
- [ ] Có classification report.
- [ ] Có confusion matrix.
- [ ] Có phân tích ảnh dự đoán sai.
- [ ] Đã lưu notebook sau khi chạy xong.